In [1]:

from pulp import *
from itertools import combinations
import numpy as np
import pickle
import random


In [2]:
# @title Class "Graph" and random graphs generation

class Graph:
    def __init__(self, number_of_nodes, edges):
        self.number_of_nodes = number_of_nodes
        self.edges = [tuple(sorted(edge)) for edge in edges]
        self.degrees = np.zeros(number_of_nodes, dtype=int)
        for edge in edges:
            self.degrees[edge[0]] += 1
            self.degrees[edge[1]] += 1
        self.neighbors = {i: set() for i in range(number_of_nodes)}
        for edge in edges:
            self.neighbors[edge[0]].add(edge[1])
            self.neighbors[edge[1]].add(edge[0])

    def __len__(self):
        return self.number_of_nodes

    def has_isolated_vertices(self):
        return np.any(self.degrees == 0)

    def greedy_clique_partition(self):
        cliques = []
        leftover_nodes = (-self.degrees).argsort().tolist()

        while leftover_nodes:
            clique_center, leftover_nodes = leftover_nodes[0], leftover_nodes[1:]
            clique = {clique_center}
            neighbors = self.neighbors[clique_center].intersection(leftover_nodes)
            densest_neighbors = sorted(neighbors, key=lambda x: -self.degrees[x])
            for neighbor in densest_neighbors:
                if all([neighbor in self.neighbors[clique_node] for clique_node in clique]):
                    clique.add(neighbor)
            cliques.append(clique)
            leftover_nodes = [node for node in leftover_nodes if node not in clique]

        return cliques

    def __str__(self):
        result = f"Graph with {self.number_of_nodes} nodes and {len(self.edges)} edges\n"
        result += f"Edges:\n"
        result += ", ".join([f"({edge[0]}, {edge[1]})" for edge in self.edges]) + "\n"
        result += "Degrees:\n"
        result += ", ".join([f"Node {i}: {degree}" for i, degree in enumerate(self.degrees)]) + "\n"
        result += "Neighbors:\n"
        for node, neighbors in self.neighbors.items():
            result += f"Node {node}: {sorted(neighbors)}\n"
        return result


def generate_random_graph(num_vertices, difficulty='medium'):
    """Generate a random graph and return a Graph object."""
    # Set edge probability based on difficulty
    edge_prob = {'easy': 0.2, 'medium': 0.4, 'hard': 0.7}.get(difficulty.lower(), 0.4)

    edges = []
    # Generate random edges
    for i in range(num_vertices):
        for j in range(i + 1, num_vertices):
            if random.random() < edge_prob:
                edges.append((i, j))

    graph = Graph(num_vertices, edges)
    return graph


def generate_graph_problems(num_graphs, min_vertices=10, max_vertices=30, difficulty='medium'):
    """Generate a list of random graph problems."""
    problems = []
    for _ in range(num_graphs):
        size = random.randint(min_vertices, max_vertices)
        graph = generate_random_graph(size, difficulty)

        # Skip graphs with isolated vertices
        if graph.has_isolated_vertices():
            continue

        problems.append(graph)
    return problems

In [3]:
# @title Solving RMP and SP
def masterSolveDual(colorings, num_vertices):
    """
    Solve the dual of the master problem (Set Covering)
    Returns: Dual values (shadow prices) for each vertex constraint
    """
    # Create the dual master LP
    master_dual = LpProblem("MasterDual", LpMaximize)

    # Decision variables - one for each vertex (dual variables)
    pi = LpVariable.dicts("pi", range(num_vertices))

    # Objective: Maximize sum of dual variables
    master_dual += lpSum(pi[v] for v in range(num_vertices))

    # Constraints: For each coloring, sum of dual vars <= 1
    for i in range(len(colorings)):
        master_dual += lpSum(pi[v] for v in range(num_vertices)
                                if colorings[i][v] == 1) <= 1

    lp = {
        'c': np.array(  [1 for v in range(num_vertices)]),
        'A': np.stack([colorings[i] for i in range(len(colorings))], axis=0),
        'b': np.array([1 for v in range(len(colorings))]),
    }

    master_dual.solve(PULP_CBC_CMD(msg=False))
    duals = [pi[v].value() for v in range(num_vertices)]
    return duals,value(master_dual.objective),lp

def masterSolve(colorings, num_vertices, relax=True):
    """
    Solve the master problem (Set Covering)
    Returns: Dual values for each vertex constraint
    """
    # Create the master LP
    master = LpProblem("Master", LpMinimize)

    # Decision variables - one for each coloring
    if relax:
        x = LpVariable.dicts("x", range(len(colorings)), lowBound=0)
    else:
        x = LpVariable.dicts("x", range(len(colorings)), lowBound=0, cat='Integer')

    # Objective: Minimize number of colorings used
    master += lpSum(x[i] for i in range(len(colorings)))

    # Constraints: Each vertex must be colored exactly once
    for v in range(num_vertices):
        constraint_name = f"Vertex_{v}"
        master += (lpSum(x[i] for i in range(len(colorings))
                           if colorings[i][v] == 1) == 1,
                    constraint_name)

    master.solve(PULP_CBC_CMD(msg=False))

    if relax:
        # Get dual values for each vertex constraint
        duals = [master.constraints[f'Vertex_{v}'].pi for v in range(num_vertices)]
        return duals
    return None

def subSolve(colorings, duals, inequalities, num_vertices,tolerance=1e-6):
    """
    Solve the subproblem (Maximum Weight Independent Set)
    Returns: Updated colorings list and boolean indicating if new coloring found
    """
    # Create subproblem
    sub = LpProblem("Sub", LpMaximize)

    # Decision variables for each vertex (1 if in independent set, 0 otherwise)
    y = LpVariable.dicts("y", range(num_vertices), cat='Binary')

    # Objective: Maximize sum of dual values
    sub += lpSum(duals[v] * y[v] for v in range(num_vertices))

    # Constraints: Adjacent vertices can't both be in independent set
    for group in inequalities:
        sub += lpSum(y[v] for v in group) <= 1

    sub.solve(PULP_CBC_CMD(msg=False))

    # If reduced cost > 1, we found a new coloring
    objective_value = value(sub.objective)
    relative_gap = abs(objective_value - 1) / max(1, abs(objective_value))

    if relative_gap > tolerance:
        try:
            new_coloring = [int(value(y[v])) for v in range(num_vertices)]
            colorings.append(new_coloring)
            return colorings, True,
        except:
            return colorings, False
    return colorings, False

In [4]:
# @title Graph coloring procedures
def get_Ab(matrix, num_cols):
    G = Graph(num_cols, matrix)

    cliques = G.greedy_clique_partition()
    inequalities = set(G.edges)
    for clique in cliques:
        clique = tuple(sorted(clique))
        for edge in combinations(clique, 2):
            inequalities.remove(edge)
        if len(clique) > 1:
            inequalities.add(clique)

    # Put trivial inequalities for nodes that didn't appear
    # in the constraints, otherwise SCIP will complain
    used_nodes = set()
    for group in inequalities:
        used_nodes.update(group)
    for node in range(num_cols):
        if node not in used_nodes:
            inequalities.add((node,))

    A, b = np.zeros((len(inequalities), len(G))), np.ones(len(inequalities))
    for ineq, group in enumerate(inequalities):
        A[ineq, sorted(group)] = 1
    return A, b, inequalities

def graphColoring(edges, num_vertices, use_dual=False):
    """Main column generation procedure for graph coloring"""
    # Initialize with singleton colorings
    colorings = [[1 if i == j else 0 for j in range(num_vertices)]
                for i in range(num_vertices)]
    A, b, inequalities = get_Ab(edges, num_vertices)
    lp_history = []
    rmp_lp_history = []

    iterations = 0
    lp = {}
    last_value = 10000
    while True:
        if use_dual:
            # Solve master problem to get dual values
            duals, value, rmp_lp = masterSolveDual(colorings, num_vertices)
            lp['master_value_delta'] = last_value - value
            lp_history.append(lp)
            rmp_lp_history.append(rmp_lp)
            last_value = value
        else:
            # Solve master problem to get dual values
            duals = masterSolve(colorings, num_vertices, relax=True)
        # Solve subproblem to find new coloring
        colorings, new_coloring = subSolve(colorings, duals, inequalities, num_vertices)

        if not new_coloring:
            break
        # Store problem data for history
        c = np.array([float(duals[v]) for v in range(num_vertices)])
        lp = {
            'c': c,
            'A': A,
            'b': b,
            'sol': new_coloring
        }
        iterations += 1

    return colorings, lp_history[1:], rmp_lp_history, A, b

def final_solve(colorings, num_vertices, max_noisy_cols,A,b):
    # Solve final integer program with generated colorings
    master = LpProblem("Graph_Coloring_IP", LpMinimize)

    # Decision variables - whether to use each coloring
    x = [LpVariable(f"x_{i}", 0, 1, LpBinary) for i in range(len(colorings))]

    # Objective - minimize number of colorings used
    master += lpSum(x)

    # Each vertex must be colored exactly once
    for v in range(num_vertices):
        master += lpSum(coloring[v] * x[i] for i, coloring in enumerate(colorings)) == 1

    # Solve integer program
    master.solve(PULP_CBC_CMD(msg=False))

    # Separate colorings based on labels
    labels = [int(x[i].varValue) for i in range(len(x))]
    used_colorings = [colorings[i] for i in range(len(colorings)) if labels[i] == 1]
    unused_colorings = [colorings[i] for i in range(len(colorings)) if labels[i] == 0]

    # Initialize lists for noisy subsets
    noisy_lists = [[] for _ in range(max_noisy_cols + 1)]

    for t_col in used_colorings:
        for i in range(max_noisy_cols):
            noisy_colorings = [col for col in used_colorings if not np.array_equal(col, t_col)]
            if len(unused_colorings) > i:
                noisy_colorings += random.sample(unused_colorings, i)

            noisy_labels = [1] * (len(used_colorings) - 1) + [0] * i
            target = [t_col]

            noisy_dict = {
                "colorings": noisy_colorings,
                "labels": noisy_labels,
                "target": target,
                "A":A,
                "b":b
            }
            noisy_lists[i].append(noisy_dict)

    # Add maximum noisy list
    for t_col in used_colorings:
        noisy_colorings = [col for col in used_colorings if not np.array_equal(col, t_col)] + unused_colorings
        noisy_labels = [1] * (len(used_colorings) - 1) + [0] * len(unused_colorings)
        target = [t_col]

        noisy_dict = {
            "colorings": noisy_colorings,
            "labels": noisy_labels,
            "target": target,
            "A":A,
            "b":b
        }
        noisy_lists[max_noisy_cols].append(noisy_dict)

    return noisy_lists

In [7]:
from tqdm import tqdm

num_graphs = 3000
min_vertices = 10
max_vertices = 30
difficulty = 'medium'
instances = generate_graph_problems(num_graphs, min_vertices, max_vertices, difficulty)

use_dual = True
max_noisy_columns = 10
fv_pairs = [[] for _ in range(max_noisy_columns+1)]

for instance in tqdm(instances):
    colorings, lp_history, rmp_lp_history, A, b = graphColoring(instance.edges, instance.number_of_nodes, use_dual=use_dual)
    noisy_lists = final_solve(colorings, instance.number_of_nodes, max_noisy_columns, A, b)

    for i in range(max_noisy_columns+1):
        fv_pairs[i].extend(noisy_lists[i])

print(len(fv_pairs[0]))
print(len(fv_pairs[1]))
print(len(fv_pairs[2]))
print(len(fv_pairs[3]))
print(len(fv_pairs[4]))
print(len(fv_pairs[5]))
print(len(fv_pairs[6]))
print(len(fv_pairs[7]))
print(len(fv_pairs[8]))
print(len(fv_pairs[9]))
print(len(fv_pairs[10]))
save_base_path = '../temp/solver_data/variations2/noise_level_'
for i, fv_list in enumerate(fv_pairs):
    if i < len(fv_pairs) - 1:
        save_path = f"{save_base_path}{i}.pkl"
    else:
        save_path = f"{save_base_path}max.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump(fv_list, f)
    print(f"Results have been saved to: {save_path}")

100%|██████████| 2958/2958 [1:28:12<00:00,  1.79s/it]  


17213
17213
17213
17213
17213
17213
17213
17213
17213
17213
17213
Results have been saved to: ../temp/solver_data/variations2/noise_level_0.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_1.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_2.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_3.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_4.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_5.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_6.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_7.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_8.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_9.pkl
Results have been saved to: ../temp/solver_data/variations2/noise_level_max.pkl


In [ ]:
from tqdm import tqdm
num_graphs = 10000
min_vertices = 10
max_vertices = 30
difficulty = 'medium'
instances = generate_graph_problems(num_graphs, min_vertices, max_vertices, difficulty)

use_dual=True
fv_pairs = []
for instance in tqdm(instances):
    colorings, lp_history, rmp_lp_history,A,b = graphColoring(instance.edges, instance.number_of_nodes, use_dual=use_dual)
    my_result = final_solve(colorings, instance.number_of_nodes)
    my_result['A'] = A
    my_result['b'] = b
    fv_pairs.append(my_result)

save_path = '../temp/graph_coloring_data.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(fv_pairs, f)
print(f"Results have been saved to: {save_path}")

100%|██████████| 9882/9882 [4:44:57<00:00,  1.73s/it]   


Results have been saved to: ../temp/graph_coloring_data.pkl
